# MCP 上下文
访问 MCP 对象内的 MCP 功能，如日志记录、进度和资源。

在定义 FastMCP工具、资源、资源模板或提示符时，您的函数可能需要与底层 MCP 会话交互或访问服务器功能。FastMCPContext为此提供了 对象。

​
## 什么是上下文？
该Context对象提供了一个干净的接口来访问函数中的 MCP 功能，包括：

- 日志记录：将调试、信息、警告和错误消息发送回客户端
- 进度报告：向客户端更新长期运行操作的进度
- 资源访问：从服务器注册的资源读取数据
- LLM 采样：请求客户的 LLM 根据提供的消息生成文本
- 请求信息：访问有关当前请求的元数据
- 服务器访问：需要时，访问底层 FastMCP 服务器实例

## 访问上下文
​
### 通过依赖注入
要在任何函数中使用上下文对象，只需在函数签名中添加一个参数，并将其类型提示为Context。FastMCP 会在调用函数时自动注入上下文实例。

### 要点：

- 参数名称（例如ctx，，context）并不重要，只有类型提示Context很重要。
- 上下文参数可以放置在函数签名中的任何位置；它不会作为有效参数暴露给 MCP 客户端。
- 上下文是可选的 - 不需要它的函数可以完全省略参数。
- 上下文方法是异步的，因此您的函数通常也需要异步。
- 类型提示可以是 union ( Context | None) 或 use Annotated[]，它仍然可以正常工作。
- 上下文仅在请求期间可用；尝试在请求之外使用上下文方法将引发错误。如果您需要在请求之外调试或调用上下文方法，您可以将变量类型设置为 ，以Context | None=None避免出现参数缺失错误。

## 工具




In [ ]:
from fastmcp import FastMCP, Context

mcp = FastMCP(name="ContextDemo")

@mcp.tool()
async def process_file(file_uri: str, ctx: Context) -> str:
    """Processes a file, using context for logging and resource access."""
    # Context is available as the ctx parameter
    return "Processed file"

## 资源和模板

In [ ]:
@mcp.resource("resource://user-data")
async def get_user_data(ctx: Context) -> dict:
    """Fetch personalized user data based on the request context."""
    # Context is available as the ctx parameter
    return {"user_id": "example"}

@mcp.resource("resource://users/{user_id}/profile")
async def get_user_profile(user_id: str, ctx: Context) -> dict:
    """Fetch user profile with context-aware logging."""
    # Context is available as the ctx parameter
    return {"id": user_id}

## 提示

In [ ]:
@mcp.prompt()
async def data_analysis_request(dataset: str, ctx: Context) -> str:
    """Generate a request to analyze data with contextual information."""
    # Context is available as the ctx parameter
    return f"Please analyze the following dataset: {dataset}"

## 通过依赖函数
虽然访问上下文的最简单方法是通过函数参数注入（如上所示），但在某些情况下，您需要在代码中访问上下文，而这些代码可能不容易修改以接受上下文参数，或者在函数调用中嵌套得更深。

FastMCP 提供了依赖函数，允许您从服务器请求执行流中的任何位置检索活动上下文：

In [ ]:
from fastmcp import FastMCP, Context
from fastmcp.server.dependencies import get_context

mcp = FastMCP(name="DependencyDemo")

# Utility function that needs context but doesn't receive it as a parameter
async def process_data(data: list[float]) -> dict:
    # Get the active context - only works when called within a request
    ctx = get_context()    
    await ctx.info(f"Processing {len(data)} data points")
    
@mcp.tool()
async def analyze_dataset(dataset_name: str) -> dict:
    # Call utility function that uses context internally
    data = load_data(dataset_name)
    await process_data(data)

重要提示：

- 该`get_context`函数只能在服务器请求上下文中使用。在请求之外调用它将引发`RuntimeError`。
- 该`get_context`函数仅适用于服务器，不应在客户端代码中使用。

## 上下文功能
​
### 日志记录
将日志消息发送回 MCP 客户端。这对于调试和查看请求期间函数的执行情况非常有用。

In [ ]:
@mcp.tool()
async def analyze_data(data: list[float], ctx: Context) -> dict:
    """Analyze numerical data with logging."""
    await ctx.debug("Starting analysis of numerical data")
    await ctx.info(f"Analyzing {len(data)} data points")
    
    try:
        result = sum(data) / len(data)
        await ctx.info(f"Analysis complete, average: {result}")
        return {"average": result, "count": len(data)}
    except ZeroDivisionError:
        await ctx.warning("Empty data list provided")
        return {"error": "Empty data list"}
    except Exception as e:
        await ctx.error(f"Analysis failed: {str(e)}")
        raise

可用的日志记录方法：

- `ctx.debug(message: str)`：有助于调试的低级细节
- `ctx.info(message: str)`：有关执行的一般信息
- `ctx.warning(message: str)`：未阻止执行的潜在问题
- `ctx.error(message: str)`：执行过程中发生的错误
- `ctx.log(level: Literal["debug", "info", "warning", "error"], message: str, logger_name: str | None = None)`：支持自定义记录器名称的通用日志方法

### 进度报告
对于长时间运行的操作，通知客户端进度。这允许客户端显示进度指示器并提供更好的用户体验。

In [ ]:
@mcp.tool()
async def process_items(items: list[str], ctx: Context) -> dict:
    """Process a list of items with progress updates."""
    total = len(items)
    results = []
    
    for i, item in enumerate(items):
        # Report progress as percentage
        await ctx.report_progress(progress=i, total=total)
        
        # Process the item (simulated with a sleep)
        await asyncio.sleep(0.1)
        results.append(item.upper())
    
    # Report 100% completion
    await ctx.report_progress(progress=total, total=total)
    
    return {"processed": len(results), "results": results}

方法签名：

- `ctx.report_progress(progress: float, total: float | None = None)`
    - progress：当前进度值（例如 24）
    - total：可选的总值（例如 100）。如果提供，客户端可以将其解释为百分比。      
    
进度报告要求客户端在初始请求中发送progressToken。如果客户端不支持进度报告，则这些调用将无效。

In [ ]:
@mcp.tool()
async def summarize_document(document_uri: str, ctx: Context) -> str:
    """Summarize a document by its resource URI."""
    # Read the document content
    content_list = await ctx.read_resource(document_uri)
    
    if not content_list:
        return "Document is empty"
    
    document_text = content_list[0].content
    
    # Example: Generate a simple summary (length-based)
    words = document_text.split()
    total_words = len(words)
    
    await ctx.info(f"Document has {total_words} words")
    
    # Return a simple summary
    if total_words > 100:
        summary = " ".join(words[:100]) + "..."
        return f"Summary ({total_words} words total): {summary}"
    else:
        return f"Full document ({total_words} words): {document_text}"

方法签名：

- `ctx.read_resource(uri: str | AnyUrl) -> list[ReadResourceContents]`
    - `uri`：要读取的资源 URI
    - 返回资源内容部分的列表（通常只包含一个项目）

返回的内容通常通过访问`content_list[0].content`，可以是文本或二进制数据，具体取决于资源。

## LLM抽样
请求客户端的 LLM 根据提供的消息生成文本。当您的函数需要利用 LLM 的功能来处理数据或生成响应时，此功能非常有用。

In [ ]:
@mcp.tool()
async def analyze_sentiment(text: str, ctx: Context) -> dict:
    """Analyze the sentiment of a text using the client's LLM."""
    # Create a sampling prompt asking for sentiment analysis
    prompt = f"Analyze the sentiment of the following text as positive, negative, or neutral. Just output a single word - 'positive', 'negative', or 'neutral'. Text to analyze: {text}"
    
    # Send the sampling request to the client's LLM
    response = await ctx.sample(prompt)
    
    # Process the LLM's response
    sentiment = response.text.strip().lower()
    
    # Map to standard sentiment values
    if "positive" in sentiment:
        sentiment = "positive"
    elif "negative" in sentiment:
        sentiment = "negative"
    else:
        sentiment = "neutral"
    
    return {"text": text, "sentiment": sentiment}

方法签名：

`ctx.sample(messages: str | list[str | SamplingMessage], system_prompt: str | None = None, temperature: float | None = None, max_tokens: int | None = None) -> TextContent | ImageContent`
- messages：要发送到 LLM 的字符串或字符串/消息对象列表
- system_prompt：可选的系统提示，用于指导 LLM 的行为
- temperature：可选采样温度（控制随机性）
- max_tokens：可选生成的最大令牌数（默认为 512）
- 将 LLM 的响应作为 TextContent 或 ImageContent 返回

当提供简单字符串时，它将被视为用户消息。对于更复杂的场景，您可以提供具有不同角色的消息列表。

In [ ]:
@mcp.tool()
async def generate_example(concept: str, ctx: Context) -> str:
    """Generate a Python code example for a given concept."""
    # Using a system prompt and a user message
    response = await ctx.sample(
        messages=f"Write a simple Python code example demonstrating '{concept}'.",
        system_prompt="You are an expert Python programmer. Provide concise, working code examples without explanations.",
        temperature=0.7,
        max_tokens=300
    )
    
    code_example = response.text
    return f"```python\n{code_example}\n```"

## 索取信息
访问有关当前请求和客户端的元数据。

In [ ]:
@mcp.tool()
async def request_info(ctx: Context) -> dict:
    """Return information about the current request."""
    return {
        "request_id": ctx.request_id,
        "client_id": ctx.client_id or "Unknown client"
    }

可用属性：

- `ctx.request_id -> str`：获取当前 MCP 请求的唯一 ID
- `ctx.client_id -> str | None`：获取发出请求的客户端的 ID（如果在初始化期间提供了）

## 高级访问
​
FastMCP 服务器和会话

In [ ]:
@mcp.tool()
async def advanced_tool(ctx: Context) -> str:
    """Demonstrate advanced context access."""
    # Access the FastMCP server instance
    server_name = ctx.fastmcp.name
    
    # Low-level session access (rarely needed)
    session = ctx.session
    request_context = ctx.request_context
    
    return f"Server: {server_name}"

### HTTP 请求
>该ctx.get_http_request()方法已弃用，并将在未来版本中移除。请改用get_http_request()依赖函数。更多详情，请参阅HTTP 请求模式。

对于 Web 应用程序，您可以访问底层 HTTP 请求：

In [ ]:
@mcp.tool()
async def handle_web_request(ctx: Context) -> dict:
    """Access HTTP request information from the Starlette request."""
    request = ctx.get_http_request()
    
    # Access HTTP headers, query parameters, etc.
    user_agent = request.headers.get("user-agent", "Unknown")
    client_ip = request.client.host if request.client else "Unknown"
    
    return {
        "user_agent": user_agent,
        "client_ip": client_ip,
        "path": request.url.path,
    }

### 高级属性参考
- `ctx.fastmcp` -> FastMCP：访问上下文所属的服务器实例
- `ctx.session`：访问原始mcp.server.session.ServerSession对象
- `ctx.request_context`：访问原始mcp.shared.context.RequestContext对象

> 直接使用`session`或`request_context`需要了解低级 `MCP Python SDK`，并且可能不如直接在Context对象上使用提供的方法稳定。